# **Store results in template**
## ID

In [ ]:
import os
import glob
import pandas as pd

# =========================
# CONFIG
# =========================
TASKS = ["addition","anagram", "geometry", "locality", "science", "logic_puzzles"]

FEATURE_COMBOS = ["emb", "adele", "emb+adele", "diff","emb+diff","emb+diff+adele","diff+adele"]

DATASETS_ROOT = "Datasets"
ASSESSOR_ROOT = "assessor_results"
FINAL_ROOT    = "Final_results"


def list_models(datasets_root: str):
    return sorted(
        d for d in os.listdir(datasets_root)
        if os.path.isdir(os.path.join(datasets_root, d))
    )


def list_prompt_csvs_for_model_task(datasets_root: str, model: str, task: str):
    """
    Returns list of (prompt_name, csv_path) for Datasets/<model>/<task>/*.csv
    """
    task_dir = os.path.join(datasets_root, model, task)
    if not os.path.isdir(task_dir):
        return []
    files = sorted(glob.glob(os.path.join(task_dir, "*.csv")))
    out = []
    for fp in files:
        prompt_name = os.path.splitext(os.path.basename(fp))[0]
        out.append((prompt_name, fp))
    return out


def find_latest_results_csv(feature_combo: str, task: str, model: str, prompt_name: str, assessor_root: str):
    """
    New layout:
      assessor_results/<feature_combo>/<task>/<model>/<prompt_name>_<timestamp>_results.csv
    """
    results_dir = os.path.join(assessor_root, feature_combo, task, model)
    pattern = os.path.join(results_dir, f"{prompt_name}_*_results.csv")
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(
            f"No assessor results for combo={feature_combo}, task={task}, model={model}, prompt={prompt_name}. "
            f"Looked for: {pattern}"
        )
    return max(files, key=os.path.getmtime)


def attach_predictions_to_original_with_all_probs(original_path: str, results_path: str):
    df_orig = pd.read_csv(original_path)
    df_res  = pd.read_csv(results_path)

    required_cols = [
        "instance_id",
        "CV_Predicted_Label",
        "CV_Predicted_Class_0",
        "CV_Predicted_Class_1",
        "CV_Predicted_Class_IDK",
    ]
    missing = [c for c in required_cols if c not in df_res.columns]
    if missing:
        raise ValueError(f"{results_path} missing cols: {missing}")

    df_merge = df_res[required_cols].rename(columns={
        "CV_Predicted_Label": "assessor_prediction",
        "CV_Predicted_Class_0": "assessor_prob_incorrect",
        "CV_Predicted_Class_1": "assessor_prob_correct",
        "CV_Predicted_Class_IDK": "assessor_prob_avoidance",
    })
    
    df_merge = df_merge.drop_duplicates(subset=["instance_id"], keep="last")

    df_out = df_orig.merge(df_merge, on="instance_id", how="left")
    return df_out


# Normalize config inputs
if isinstance(TASKS, str):
    TASKS = [TASKS]
if isinstance(FEATURE_COMBOS, str):
    FEATURE_COMBOS = [FEATURE_COMBOS]


# =========================
# RUN
# =========================
models = list_models(DATASETS_ROOT)
print("Models:", models)
print("Feature combos:", FEATURE_COMBOS)

for feature_combo in FEATURE_COMBOS:
    for task in TASKS:
        print(f"\n==============================")
        print(f"COMBO: {feature_combo} | TASK: {task}")
        print(f"==============================")

        for model in models:
            prompt_files = list_prompt_csvs_for_model_task(DATASETS_ROOT, model, task)
            if not prompt_files:
                print(f"[Skip] No prompt csvs for model={model}, task={task}")
                continue

            out_dir = os.path.join(FINAL_ROOT, feature_combo, task, model)
            os.makedirs(out_dir, exist_ok=True)

            for prompt_name, orig_csv in prompt_files:
                try:
                    res_csv = find_latest_results_csv(feature_combo, task, model, prompt_name, ASSESSOR_ROOT)
                    df_final = attach_predictions_to_original_with_all_probs(orig_csv, res_csv)

                    out_csv = os.path.join(out_dir, f"{prompt_name}.csv")
                    df_final.to_csv(out_csv, index=False)

                    print(f"[{feature_combo} | {task} | {model} | {prompt_name}] -> wrote: {out_csv}")
                    print(f"   used results: {res_csv}")

                except Exception as e:
                    print(f"[{feature_combo} | {task} | {model} | {prompt_name}] ERROR: {e}")


## OOD

In [ ]:
import os
import glob
import pandas as pd

# =========================
# CONFIG
# =========================
TASKS = ["addition", "anagram", "science", "geometry", "locality", "logic_puzzles"]

FEATURE_COMBOS = ["emb","adele", "diff", "emb+diff","diff+adele","emb+adele","emb+diff+adele"]

DATASETS_ROOT = "Datasets"
ASSESSOR_ROOT = "assessor_results_ood"
FINAL_ROOT = "Final_results"


def list_models(datasets_root: str):
    return sorted(
        d for d in os.listdir(datasets_root)
        if os.path.isdir(os.path.join(datasets_root, d))
    )


def list_prompt_csvs_for_model_task(datasets_root: str, model: str, task: str):
    """
    Returns list of (prompt_name, csv_path) for Datasets/<model>/<task>/*.csv
    """
    task_dir = os.path.join(datasets_root, model, task)
    if not os.path.isdir(task_dir):
        return []

    files = sorted(glob.glob(os.path.join(task_dir, "*.csv")))
    out = []
    for fp in files:
        prompt_name = os.path.splitext(os.path.basename(fp))[0]
        out.append((prompt_name, fp))
    return out


def find_latest_results_csv(feature_combo: str, task: str, model: str, prompt_name: str, assessor_root: str):
    """
    OOD layout:
      assessor_results_ood/<feature_combo>/leave_out_<task>/<model>/<prompt_name>_<timestamp>_results.csv
    """
    results_dir = os.path.join(assessor_root, feature_combo, f"leave_out_{task}", model)
    pattern = os.path.join(results_dir, f"{prompt_name}_*_results.csv")
    files = glob.glob(pattern)

    if not files:
        raise FileNotFoundError(
            f"No OOD assessor results for combo={feature_combo}, leave_out={task}, model={model}, prompt={prompt_name}. "
            f"Looked for: {pattern}"
        )

    return max(files, key=os.path.getmtime)


def attach_predictions_to_original_with_all_probs(original_path: str, results_path: str):
    df_orig = pd.read_csv(original_path)
    df_res = pd.read_csv(results_path)

    required_cols = [
        "instance_id",
        "OOD_Predicted_Label",
        "OOD_Predicted_Class_0",
        "OOD_Predicted_Class_1",
        "OOD_Predicted_Class_IDK",
    ]
    missing = [c for c in required_cols if c not in df_res.columns]
    if missing:
        raise ValueError(f"{results_path} missing cols: {missing}")

    df_merge = df_res[required_cols].rename(columns={
        "OOD_Predicted_Label": "assessor_prediction",
        "OOD_Predicted_Class_0": "assessor_prob_incorrect",
        "OOD_Predicted_Class_1": "assessor_prob_correct",
        "OOD_Predicted_Class_IDK": "assessor_prob_avoidance",
    })

    df_merge = df_merge.drop_duplicates(subset=["instance_id"], keep="last")

    df_out = df_orig.merge(df_merge, on="instance_id", how="left")
    return df_out


if isinstance(TASKS, str):
    TASKS = [TASKS]
if isinstance(FEATURE_COMBOS, str):
    FEATURE_COMBOS = [FEATURE_COMBOS]


# =========================
# RUN
# =========================
models = list_models(DATASETS_ROOT)
print("Models:", models)
print("Feature combos:", FEATURE_COMBOS)

for feature_combo in FEATURE_COMBOS:
    for task in TASKS:
        print(f"\n==============================")
        print(f"COMBO: {feature_combo} | HELD-OUT TASK: {task}")
        print(f"==============================")

        for model in models:
            prompt_files = list_prompt_csvs_for_model_task(DATASETS_ROOT, model, task)
            if not prompt_files:
                print(f"[Skip] No prompt csvs for model={model}, task={task}")
                continue

            out_dir = os.path.join(FINAL_ROOT, feature_combo, f"leave_out_{task}", model)
            os.makedirs(out_dir, exist_ok=True)

            for prompt_name, orig_csv in prompt_files:
                try:
                    res_csv = find_latest_results_csv(feature_combo, task, model, prompt_name, ASSESSOR_ROOT)
                    df_final = attach_predictions_to_original_with_all_probs(orig_csv, res_csv)

                    out_csv = os.path.join(out_dir, f"{prompt_name}.csv")
                    df_final.to_csv(out_csv, index=False)

                    print(f"[{feature_combo} | leave_out_{task} | {model} | {prompt_name}] -> wrote: {out_csv}")
                    print(f"   used results: {res_csv}")

                except Exception as e:
                    print(f"[{feature_combo} | leave_out_{task} | {model} | {prompt_name}] ERROR: {e}")

Models: ['GPT5.2', 'Qwen2.5_72B', 'gemini-3-pro-preview', 'llama3v3chat-70B']
Feature combos: ['emb', 'adele', 'diff', 'emb+diff', 'diff+adele', 'emb+adele', 'emb+diff+adele']

COMBO: emb | HELD-OUT TASK: addition
[emb | leave_out_addition | GPT5.2 | CoT] -> wrote: Final_results/emb/leave_out_addition/GPT5.2/CoT.csv
   used results: assessor_results_ood/emb/leave_out_addition/GPT5.2/CoT_2026-04-20_02-45-58_results.csv
[emb | leave_out_addition | GPT5.2 | multi_step] -> wrote: Final_results/emb/leave_out_addition/GPT5.2/multi_step.csv
   used results: assessor_results_ood/emb/leave_out_addition/GPT5.2/multi_step_2026-04-20_02-50-50_results.csv
[emb | leave_out_addition | GPT5.2 | self_probing] -> wrote: Final_results/emb/leave_out_addition/GPT5.2/self_probing.csv
   used results: assessor_results_ood/emb/leave_out_addition/GPT5.2/self_probing_2026-04-20_02-55-36_results.csv
[emb | leave_out_addition | GPT5.2 | top_k] -> wrote: Final_results/emb/leave_out_addition/GPT5.2/top_k.csv
   use